In [22]:
# 建立模型结构

import sympy as sp


print("开始建立机械臂动力学模型...")

# 1. 定义符号
# q1: yaw, q2: pitch1, q3: pitch2
q1, q2, q3 = sp.symbols('q1 q2 q3', real=True)
dq1, dq2, dq3 = sp.symbols('dq1 dq2 dq3', real=True)
ddq1, ddq2, ddq3 = sp.symbols('ddq1 ddq2 ddq3', real=True)

# 连杆质量与尺寸参数
m1, m2 = sp.symbols('m1 m2', real=True, positive=True)
L1 = sp.symbols('L1', real=True, positive=True) # 第一连杆长度
d1, d2 = sp.symbols('d1 d2', real=True, positive=True) # 质心距离

# 转动惯量 (假设在质心主轴上的对角阵)
I1xx, I1yy, I1zz = sp.symbols('I1xx I1yy I1zz', real=True, positive=True)
I2xx, I2yy, I2zz = sp.symbols('I2xx I2yy I2zz', real=True, positive=True)

# 动态等效重力向量 (用于补偿基座的三维平动加速度, g_app = g_world - a_base)
gx, gy, gz = sp.symbols('gx gy gz', real=True)
g_vec = sp.Matrix([gx, gy, gz])

# 状态向量
q = sp.Matrix([q1, q2, q3])
dq = sp.Matrix([dq1, dq2, dq3])

print("正在计算正运动学...")
# 2. 坐标系与旋转矩阵
# 基坐标系：X向前，Y向左，Z向上
# 偏航关节 (Yaw, q1) 绕Z轴旋转
R_z = sp.Matrix([
    [sp.cos(q1), -sp.sin(q1), 0],
    [sp.sin(q1),  sp.cos(q1), 0],
    [0,           0,          1]
])

# 俯仰关节 (Pitch)。零位均指向+X方向。
# 规定：q为正时，机械臂抬起 (偏向+Z方向)。
# 根据右手法则，平面向Z轴正向旋转对应绕基坐标系负Y轴 (-Y) 的旋转。
def rot_y_neg(th):
    return sp.Matrix([
        [sp.cos(th), 0, -sp.sin(th)],
        [0,          1,  0         ],
        [sp.sin(th), 0,  sp.cos(th)]
    ])

R_p1 = rot_y_neg(q2)
R_p2 = rot_y_neg(q3)
R_p12 = rot_y_neg(q2 + q3)

# 连杆1与连杆2的全局旋转矩阵
R1 = R_z * R_p1
R2 = R_z * R_p12

# 质心位置 (Center of Mass)
# p_joint12 = [0,0,0]
p_c1 = R1 * sp.Matrix([d1, 0, 0])
p_joint3 = R1 * sp.Matrix([L1, 0, 0])
p_c2 = p_joint3 + R2 * sp.Matrix([d2, 0, 0])

print("正在计算平动与转动速度...")
# 3. 速度与动能
# 平动求导
v_c1 = p_c1.jacobian(q) * dq
v_c2 = p_c2.jacobian(q) * dq

# 角速度
# 偏航轴方向永远是 [0, 0, 1]^T
# 俯仰轴方向是绕Z轴yaw后的 -Y轴，即 R_z * [0, -1, 0]^T
omega1_global = sp.Matrix([0, 0, dq1]) + R_z * sp.Matrix([0, -dq2, 0])
omega2_global = sp.Matrix([0, 0, dq1]) + R_z * sp.Matrix([0, -dq2 - dq3, 0])

# 将角速度转换到各自连杆的局部坐标系下，以使用对角化的惯量矩阵
omega1_local = R1.T * omega1_global
omega2_local = R2.T * omega2_global

I1 = sp.diag(I1xx, I1yy, I1zz)
I2 = sp.diag(I2xx, I2yy, I2zz)

# 动能 (平动 + 转动)
T_trans = 0.5 * m1 * v_c1.dot(v_c1) + 0.5 * m2 * v_c2.dot(v_c2)
T_rot = 0.5 * (omega1_local.T * I1 * omega1_local)[0] + 0.5 * (omega2_local.T * I2 * omega2_local)[0]
T = sp.simplify(T_trans + T_rot)

# 势能
# U = - m * g_vec^T * p_c (g_vec 指向重力方向, 如 [0,0,-g]^T)
print("正在计算势能...")
V = - m1 * g_vec.dot(p_c1) - m2 * g_vec.dot(p_c2)
V = sp.simplify(V)

print("正在推导质量矩阵 M...")
# 4. 提取矩阵 M(q)
M = sp.zeros(3, 3)
for i in range(3):
    for j in range(3):
        # 动能对速度的二阶偏导为 M_ij
        M[i, j] = sp.diff(sp.diff(T, dq[i]), dq[j])
M = sp.simplify(M)

print("正在推导科里奥利矩阵 C...")
# 5. 提取科里奥利与离心力矩阵 C(q, dq) 使用 Christoffel symbol 
C = sp.zeros(3, 3)
for k in range(3):
    for j in range(3):
        c_val = 0
        for i in range(3):
            # c_ijk = 1/2 * ( dM_kj/dqi + dM_ki/dqj - dM_ij/dqk )
            c_ijk = 0.5 * (sp.diff(M[k, j], q[i]) + sp.diff(M[k, i], q[j]) - sp.diff(M[i, j], q[k]))
            c_val += c_ijk * dq[i]
        C[k, j] = c_val
C = sp.simplify(C)

print("正在推导重力项 G...")
# 6. 计算重力项 G(q)
G = sp.zeros(3, 1)
for i in range(3):
    G[i] = sp.diff(V, q[i])
G = sp.simplify(G)


开始建立机械臂动力学模型...
正在计算正运动学...
正在计算平动与转动速度...
正在计算势能...
正在推导质量矩阵 M...
正在推导科里奥利矩阵 C...
正在推导重力项 G...


In [23]:
# ============================================================
# 7. 摩擦模型 + 完整动力学方程
# ============================================================
print("\n=== 定义摩擦模型与完整动力学 ===\n")

# 库仑摩擦 + 粘滞摩擦参数 (数值辨识阶段使用)
fc1, fc2, fc3 = sp.symbols('fc1 fc2 fc3', real=True)  # Coulomb friction coefficient
fv1, fv2, fv3 = sp.symbols('fv1 fv2 fv3', real=True)  # Viscous friction coefficient

# 摩擦转矩 (符号形式)
# τ_f_i = fc_i * sign(dq_i) + fv_i * dq_i
tau_f_sym = sp.Matrix([
    fc1 * sp.sign(dq1) + fv1 * dq1,
    fc2 * sp.sign(dq2) + fv2 * dq2,
    fc3 * sp.sign(dq3) + fv3 * dq3,
])

# 完整无摩擦刚体动力学: τ_inertial = M(q)*ddq + C(q,dq)*dq + G(q)
ddq_vec = sp.Matrix([ddq1, ddq2, ddq3])
tau_inertial = M * ddq_vec + C * dq + G
tau_inertial = sp.simplify(tau_inertial)

# 含摩擦的完整动力学
tau_full = tau_inertial + tau_f_sym
tau_full = sp.simplify(tau_full)

print("✓ 无摩擦刚体动力学 τ_inertial = M·ddq + C·dq + G 推导完成")
print("✓ 摩擦模型: 库仑 + 粘滞 (共 6 个参数)")
print(f"  τ_i = (M·ddq + C·dq + G)_i + fc_i·sign(dq_i) + fv_i·dq_i")
print(f"\nM 矩阵维度: {M.shape}")
print(f"C 矩阵维度: {C.shape}")
print(f"G 向量维度: {G.shape}")


=== 定义摩擦模型与完整动力学 ===

✓ 无摩擦刚体动力学 τ_inertial = M·ddq + C·dq + G 推导完成
✓ 摩擦模型: 库仑 + 粘滞 (共 6 个参数)
  τ_i = (M·ddq + C·dq + G)_i + fc_i·sign(dq_i) + fv_i·dq_i

M 矩阵维度: (3, 3)
C 矩阵维度: (3, 3)
G 向量维度: (3, 1)


In [61]:
# ============================================================
# Phase 1: 数据加载与预处理
# ============================================================
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt

print("=== Phase 1: 数据加载与预处理 ===\n")

# --- 1. 读取实验数据 ---
DATA_FILE = "data_20260514_130938.csv"
df = pd.read_csv(DATA_FILE)
print(f"加载数据文件: {DATA_FILE}")
print(f"总样本数: {len(df)}")
print(f"列名: {list(df.columns)}")

# 提取时间戳与状态
t_raw = df["stamp"].values
q_raw = df[["q1", "q2", "q3"]].values      # (N, 3)
dq_raw = df[["dq1", "dq2", "dq3"]].values   # (N, 3)
tau_raw = df[["tau1", "tau2", "tau3"]].values

# --- 2. 计算采样间隔 ---
dt = float(np.median(np.diff(t_raw)))
fs = 1.0 / dt
print(f"采样间隔 dt = {dt:.4f}s, 采样率 fs = {fs:.1f}Hz")

# --- 3. 计算加速度 (中心差分) ---
N = len(t_raw)
ddq_raw = np.zeros_like(dq_raw)
for j in range(3):
    ddq_raw[1:-1, j] = (dq_raw[2:, j] - dq_raw[:-2, j]) / (2.0 * dt)
# 边界用单侧差分
ddq_raw[0, :] = (dq_raw[1, :] - dq_raw[0, :]) / dt
ddq_raw[-1, :] = (dq_raw[-1, :] - dq_raw[-2, :]) / dt

# --- 4. 零相位低通滤波 ---
fc = 5.0  # 截止频率 Hz
order = 4
b, a = butter(order, fc / (fs / 2), btype='low')

def zero_phase_filter(x, b, a):
    """对每列做零相位滤波"""
    return np.column_stack([filtfilt(b, a, x[:, j]) for j in range(x.shape[1])])

q_filt   = zero_phase_filter(q_raw, b, a)
dq_filt  = zero_phase_filter(dq_raw, b, a)
ddq_filt = zero_phase_filter(ddq_raw, b, a)
tau_filt = zero_phase_filter(tau_raw, b, a)

# --- 5. 剔除首尾边界效应 ---
n_trim = 10
q_filt   = q_filt[n_trim:-n_trim, :]
dq_filt  = dq_filt[n_trim:-n_trim, :]
ddq_filt = ddq_filt[n_trim:-n_trim, :]
tau_filt = tau_filt[n_trim:-n_trim, :]
t = t_raw[n_trim:-n_trim]
N = len(t)

# --- 6. 打印数据概况 ---
print(f"\n滤波+修剪后样本数: N = {N}")
for j, name in enumerate(["q1","q2","q3"]):
    print(f"  {name}: range=[{q_filt[:,j].min():.3f}, {q_filt[:,j].max():.3f}] rad, "
          f"|dq|_max={np.abs(dq_filt[:,j]).max():.3f} rad/s, "
          f"|tau|_max={np.abs(tau_filt[:,j]).max():.2f} Nm")
print("\n✓ 数据预处理完成")

=== Phase 1: 数据加载与预处理 ===

加载数据文件: data_20260514_130938.csv
总样本数: 4384
列名: ['stamp', 'q1', 'q2', 'q3', 'dq1', 'dq2', 'dq3', 'tau1', 'tau2', 'tau3']
采样间隔 dt = 0.0500s, 采样率 fs = 20.0Hz

滤波+修剪后样本数: N = 4364
  q1: range=[-0.457, 0.156] rad, |dq|_max=1.166 rad/s, |tau|_max=3.00 Nm
  q2: range=[1.800, 2.540] rad, |dq|_max=1.010 rad/s, |tau|_max=26.30 Nm
  q3: range=[-2.122, -1.017] rad, |dq|_max=1.746 rad/s, |tau|_max=22.13 Nm

✓ 数据预处理完成


In [65]:
# ============================================================
# Phase 2: 固定已知参数 + 提取惯性回归矩阵
# ============================================================
import sympy as sp

print("=== Phase 2: 固定已知参数，提取惯性回归矩阵 ===\n")

# --- 1. 固定已知质量/尺寸/重力参数 ---
L1_val = 0.4235   # m
m1_val = 1.8      # kg
m2_val = 3.35     # kg
d1_val = 0.175    # m
d2_val = 0.330    # m
g_val  = 9.81     # m/s²

geo_grav_subs = {
    m1: m1_val, m2: m2_val,
    d1: d1_val, d2: d2_val,
    L1: L1_val,
    gx: 0, gy: 0, gz: -g_val,
}
print("固定参数:")
for k, v in geo_grav_subs.items():
    print(f"  {k} = {v}")

# 代入到无摩擦刚体动力学 τ_inertial = M·ddq + C·dq + G
tau_inertial_sub = sp.simplify(tau_inertial.subs(geo_grav_subs))
print(f"\nτ_inertial 代入后表达式已化简")

# --- 2. 定义惯性参数向量 ---
inertia_params = [I1xx, I1yy, I1zz, I2xx, I2yy, I2zz]
inertia_names  = ['I1xx', 'I1yy', 'I1zz', 'I2xx', 'I2yy', 'I2zz']
n_inertia = len(inertia_params)
print(f"待辨识惯性参数 ({n_inertia} 个): {inertia_names}")

# --- 3. 构建惯性回归矩阵 Y_inertial_sym (3 × 6) ---
# τ_inertial 对惯性参数是线性的: τ_inertial_sub = Y_inertial · θ_inertial
Y_inertial_sym = sp.zeros(3, n_inertia)
for i in range(3):
    for j in range(n_inertia):
        Y_inertial_sym[i, j] = sp.simplify(
            sp.diff(tau_inertial_sub[i], inertia_params[j])
        )
print("✓ 惯性回归矩阵 Y_inertial_sym (3×6) 构建完成")

# --- 4. 数值化: 对每个 joint 单独 lambdify 标量表达式 ---
tau_funcs = []
for i in range(3):
    f = sp.lambdify(
        (q1, q2, q3, dq1, dq2, dq3, ddq1, ddq2, ddq3,
         I1xx, I1yy, I1zz, I2xx, I2yy, I2zz),
        tau_inertial_sub[i], "numpy"
    )
    tau_funcs.append(f)

def tau_inertial_numeric(state_vec, theta_vec):
    """计算单个样本点的 τ_inertial (3,)
    state_vec: [q1, q2, q3, dq1, dq2, dq3, ddq1, ddq2, ddq3]
    theta_vec: [I1xx, I1yy, I1zz, I2xx, I2yy, I2zz]
    """
    args = (*state_vec, *theta_vec)
    return np.array([f(*args) for f in tau_funcs], dtype=float)

def compute_Y_inertial(q_arr, dq_arr, ddq_arr):
    """
    通过数值雅可比批量计算惯性回归矩阵
    """
    Ns = len(q_arr)
    eps = 1e-6
    theta0 = np.zeros(n_inertia)
    Y_blocks = []
    for k in range(Ns):
        state = [q_arr[k,0], q_arr[k,1], q_arr[k,2],
                 dq_arr[k,0], dq_arr[k,1], dq_arr[k,2],
                 ddq_arr[k,0], ddq_arr[k,1], ddq_arr[k,2]]
        Yk = np.zeros((3, n_inertia))
        for j in range(n_inertia):
            tp = theta0.copy(); tp[j] = eps
            Yk[:, j] = tau_inertial_numeric(state, tp) / eps
        Y_blocks.append(Yk)
    return np.vstack(Y_blocks)

# 批量计算
print(f"\n正在批量计算 Y_inertial (样本数 N = {N})...")
Y_inertial = compute_Y_inertial(q_filt, dq_filt, ddq_filt)
print(f"Y_inertial 形状: {Y_inertial.shape}")  # 应为 (3*N, 6)

# --- 5. 数值验证：线性化正确性 ---
# 用随机参数测试: |Y_inertial @ theta_test - tau_inertial_sub(theta_test)| < 1e-6
np.random.seed(42)
theta_test = np.array([0.01, 0.02, 0.03, 0.04, 0.05, 0.06])

# 随机抽取10个样本点验证
idx_test = np.linspace(0, N-1, 10, dtype=int)
max_err = 0.0
for k in idx_test:
    state_k = [q_filt[k,0], q_filt[k,1], q_filt[k,2],
               dq_filt[k,0], dq_filt[k,1], dq_filt[k,2],
               ddq_filt[k,0], ddq_filt[k,1], ddq_filt[k,2]]
    tau_direct = tau_inertial_numeric(state_k, theta_test)
    # 用数值雅可比计算 Yk
    eps = 1e-6
    theta0 = np.zeros(n_inertia)
    Yk = np.zeros((3, n_inertia))
    for j in range(n_inertia):
        tp = theta0.copy(); tp[j] = eps
        Yk[:, j] = tau_inertial_numeric(state_k, tp) / eps
    tau_lin = Yk @ theta_test
    err = np.max(np.abs(tau_direct - tau_lin))
    max_err = max(max_err, err)

print(f"线性化验证: 最大误差 = {max_err:.2e}")
assert max_err < 1e-6, f"线性化误差过大: {max_err}"
print("✓ 惯性回归矩阵线性化验证通过")


=== Phase 2: 固定已知参数，提取惯性回归矩阵 ===

固定参数:
  m1 = 1.8
  m2 = 3.35
  d1 = 0.175
  d2 = 0.33
  L1 = 0.4235
  gx = 0
  gy = 0
  gz = -9.81

τ_inertial 代入后表达式已化简
待辨识惯性参数 (6 个): ['I1xx', 'I1yy', 'I1zz', 'I2xx', 'I2yy', 'I2zz']
✓ 惯性回归矩阵 Y_inertial_sym (3×6) 构建完成


SyntaxError: cannot assign to expression (<lambdifygenerated-33>, line 2)

In [ ]:
# ============================================================
# Phase 3: 构建完整回归矩阵 + 网格搜索超参 α₁, α₃
# ============================================================
from sklearn.linear_model import RidgeCV

print("=== Phase 3: 完整回归矩阵 + 网格搜索 ===\n")

# --- 1. 目标向量: 实测力矩扁平化 ---
# 堆叠顺序: [tau1_0..tau1_{N-1}, tau2_0..tau2_{N-1}, tau3_0..tau3_{N-1}]
tau_flat = tau_filt.flatten(order='F')  # (3*N,)  Fortran order: joint-major
print(f"目标向量 tau_flat 形状: {tau_flat.shape}")

# --- 2. 构建摩擦特征列的辅助函数 ---
# Y_fric 是 (3N × 6) 的块对角结构:
#   [fc1_col, fv1_col, 0,      0,      0,      0     ]  ← rows 0..N-1 (joint1)
#   [0,       0,       fc2_col, fv2_col, 0,      0     ]  ← rows N..2N-1 (joint2)
#   [0,       0,       0,       0,       fc3_col, fv3_col]  ← rows 2N..3N-1 (joint3)

def build_Y_fric(dq_data, alpha1, alpha3):
    """
    dq_data: (N, 3) 滤波后的关节速度
    alpha1, alpha3: tanh 陡峭系数 (joint2 用 sign)
    返回: Y_fric (3*N, 6)
    """
    Ns = len(dq_data)
    Y_f = np.zeros((3 * Ns, 6))
    
    # Joint 1: tanh(alpha1 * dq1)  +  dq1
    Y_f[:Ns, 0] = np.tanh(alpha1 * dq_data[:, 0])  # fc1
    Y_f[:Ns, 1] = dq_data[:, 0]                      # fv1
    
    # Joint 2: sign(dq2)  +  dq2
    Y_f[Ns:2*Ns, 2] = np.sign(dq_data[:, 1])         # fc2
    Y_f[Ns:2*Ns, 3] = dq_data[:, 1]                   # fv2
    
    # Joint 3: tanh(alpha3 * dq3)  +  dq3
    Y_f[2*Ns:3*Ns, 4] = np.tanh(alpha3 * dq_data[:, 2])  # fc3
    Y_f[2*Ns:3*Ns, 5] = dq_data[:, 2]                     # fv3
    
    return Y_f

# --- 3. 网格搜索 ---
alpha_grid = [10, 20, 50, 100, 200, 500]
ridge_alphas = np.logspace(-3, 3, 20)  # Ridge 正则化候选

print(f"网格搜索: α₁, α₃ ∈ {alpha_grid}")
print(f"RidgeCV alphas: {ridge_alphas[0]:.3f} ~ {ridge_alphas[-1]:.1f}")
print(f"{'α₁':>6} {'α₃':>6} {'R²':>8} {'Ridge α':>10}")
print("-" * 35)

best_r2 = -np.inf
best_alpha1 = None
best_alpha3 = None
best_ridge_alpha = None
best_theta = None

for alpha1 in alpha_grid:
    for alpha3 in alpha_grid:
        Y_fric = build_Y_fric(dq_filt, alpha1, alpha3)
        Y_full = np.hstack([Y_inertial, Y_fric])  # (3N, 12)
        
        ridge = RidgeCV(alphas=ridge_alphas, fit_intercept=False,
                        scoring='r2', cv=5)
        ridge.fit(Y_full, tau_flat)
        
        r2 = ridge.score(Y_full, tau_flat)
        print(f"{alpha1:>6} {alpha3:>6} {r2:>8.4f} {ridge.alpha_:>10.3f}")
        
        if r2 > best_r2:
            best_r2 = r2
            best_alpha1 = alpha1
            best_alpha3 = alpha3
            best_ridge_alpha = ridge.alpha_
            best_theta = ridge.coef_.copy()

print(f"\n✓ 最佳超参: α₁* = {best_alpha1}, α₃* = {best_alpha3}, Ridge α* = {best_ridge_alpha:.3f}, R² = {best_r2:.4f}")


In [ ]:
# ============================================================
# Phase 4: 最优参数辨识与评估
# ============================================================
print("=== Phase 4: 最优参数辨识与评估 ===\n")

# --- 1. 最终拟合 ---
Y_fric_opt = build_Y_fric(dq_filt, best_alpha1, best_alpha3)
Y_full_opt = np.hstack([Y_inertial, Y_fric_opt])

ridge_final = RidgeCV(alphas=ridge_alphas, fit_intercept=False, scoring='r2', cv=5)
ridge_final.fit(Y_full_opt, tau_flat)
theta_hat = ridge_final.coef_

# --- 2. 参数分解 ---
theta_inertial = theta_hat[:6]   # [I1xx, I1yy, I1zz, I2xx, I2yy, I2zz]
theta_fric     = theta_hat[6:]   # [fc1, fv1, fc2, fv2, fc3, fv3]

print("="*55)
print("  辨识参数结果")
print("="*55)
print("\n--- 惯量参数 (kg·m²) ---")
for name, val in zip(inertia_names, theta_inertial):
    flag = " ✓" if val > 0 else " ✗ (检查!)"
    print(f"  {name:6s} = {val:12.6f}{flag}")

print("\n--- 摩擦参数 ---")
fric_names = ['fc1', 'fv1', 'fc2', 'fv2', 'fc3', 'fv3']
fric_units = ['Nm', 'Nm·s/rad', 'Nm', 'Nm·s/rad', 'Nm', 'Nm·s/rad']
for name, val, unit in zip(fric_names, theta_fric, fric_units):
    if 'fc' in name:
        flag = " ✓" if val > 0 else " ✗ (Coulomb<0!)"
    else:
        flag = ""
    print(f"  {name:6s} = {val:+12.6f} {unit:12s}{flag}")

# --- 3. 预测力矩 ---
tau_pred_flat = Y_full_opt @ theta_hat
tau_pred = tau_pred_flat.reshape(3, N, order='F').T  # (N, 3)

# --- 4. 评估指标 ---
print("\n" + "="*55)
print("  评估指标 (逐关节 + 总体)")
print("="*55)
print(f"{'Joint':>8} {'R²':>8} {'RMSE(Nm)':>10} {'NRMSE(%)':>10} {'τ_range(Nm)':>12}")

r2_list, rmse_list, nrmse_list = [], [], []
for j in range(3):
    ss_res = np.sum((tau_filt[:, j] - tau_pred[:, j]) ** 2)
    ss_tot = np.sum((tau_filt[:, j] - np.mean(tau_filt[:, j])) ** 2)
    r2_j = 1.0 - ss_res / ss_tot
    rmse_j = np.sqrt(ss_res / N)
    tau_range = tau_filt[:, j].max() - tau_filt[:, j].min()
    nrmse_j = rmse_j / tau_range * 100 if tau_range > 0 else 0.0
    r2_list.append(r2_j)
    rmse_list.append(rmse_j)
    nrmse_list.append(nrmse_j)
    print(f"{'tau'+str(j+1):>8} {r2_j:>8.4f} {rmse_j:>10.4f} {nrmse_j:>10.2f} {tau_range:>12.2f}")

# 总体
ss_res_all = np.sum((tau_filt - tau_pred) ** 2)
ss_tot_all = np.sum((tau_filt - np.mean(tau_filt)) ** 2)
r2_all = 1.0 - ss_res_all / ss_tot_all
rmse_all = np.sqrt(ss_res_all / (3 * N))
print(f"{'Overall':>8} {r2_all:>8.4f} {rmse_all:>10.4f}")

# --- 5. 保存结果 ---
# 评估指标
df_metrics = pd.DataFrame({
    'Joint': ['tau1', 'tau2', 'tau3', 'Overall'],
    'R²': r2_list + [r2_all],
    'RMSE_Nm': rmse_list + [rmse_all],
    'NRMSE_pct': nrmse_list + [np.nan],
    'TorqueRange_Nm': [tau_filt[:,j].max()-tau_filt[:,j].min() for j in range(3)] + [np.nan]
})
df_metrics.to_csv('identification_metrics.csv', index=False)
print("\n✓ 指标已保存至 identification_metrics.csv")

# 预测 vs 实测
df_pred = pd.DataFrame({
    'time_s': t,
    'tau1_meas': tau_filt[:, 0], 'tau1_pred': tau_pred[:, 0],
    'tau2_meas': tau_filt[:, 1], 'tau2_pred': tau_pred[:, 1],
    'tau3_meas': tau_filt[:, 2], 'tau3_pred': tau_pred[:, 2],
})
df_pred.to_csv('prediction_vs_measurement.csv', index=False)
print("✓ 预测数据已保存至 prediction_vs_measurement.csv")
print("\n✓ 参数辨识与评估完成")


In [ ]:
# ============================================================
# Phase 5: 可视化
# ============================================================
import matplotlib.pyplot as plt

print("=== Phase 5: 可视化 ===\n")

# 设置中文字体 (若不可用则回退)
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
colors = ['#1f77b4', '#d62728']

for j in range(3):
    ax = axes[j]
    ax.plot(t, tau_filt[:, j], color=colors[0], linewidth=0.8, label='Measured')
    ax.plot(t, tau_pred[:, j], color=colors[1], linewidth=0.8, linestyle='--', label='Predicted')
    ax.set_ylabel(f'τ{j+1} (Nm)', fontsize=11)
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.text(0.02, 0.92, f'R²={r2_list[j]:.3f}  RMSE={rmse_list[j]:.3f} Nm',
            transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

axes[-1].set_xlabel('Time (s)', fontsize=12)
fig.suptitle('Dynamics Identification: Measured vs Predicted Torque', fontsize=14)
plt.tight_layout()
plt.savefig('torque_prediction.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ 力矩预测图已保存至 torque_prediction.png")

# --- 残差图 ---
fig2, axes2 = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
for j in range(3):
    residual = tau_filt[:, j] - tau_pred[:, j]
    ax = axes2[j]
    ax.plot(t, residual, color='#9467bd', linewidth=0.6)
    ax.axhline(y=0, color='gray', linestyle=':', linewidth=0.8)
    ax.set_ylabel(f'Residual τ{j+1} (Nm)', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.text(0.02, 0.85, f'std={np.std(residual):.3f} Nm',
            transform=ax.transAxes, fontsize=9)

axes2[-1].set_xlabel('Time (s)', fontsize=12)
fig2.suptitle('Residuals: τ_meas - τ_pred', fontsize=14)
plt.tight_layout()
plt.savefig('torque_residuals.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ 残差图已保存至 torque_residuals.png")

# --- SVD 分析惯性回归矩阵 ---
U, S, Vt = np.linalg.svd(Y_inertial, full_matrices=False)
cond = S[0] / S[-1] if S[-1] > 0 else np.inf
# 有效秩 (奇异值 > 最大奇异值的 1%)
effective_rank = np.sum(S > 0.01 * S[0])

fig3, ax3 = plt.subplots(figsize=(10, 4))
ax3.semilogy(S, 'o-', color='#2ca02c', markersize=6)
ax3.axhline(y=0.01*S[0], color='red', linestyle='--', linewidth=1, label='1% threshold')
ax3.set_xlabel('Index', fontsize=11)
ax3.set_ylabel('Singular Value', fontsize=11)
ax3.set_title(f'SVD of Y_inertial: cond={cond:.1f}, effective_rank={effective_rank}/6', fontsize=12)
ax3.legend()
ax3.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('inertia_svd.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ SVD 分析: 条件数={cond:.1f}, 有效秩={effective_rank}/6")
print("✓ SVD 图已保存至 inertia_svd.png")

print("\n✓ 全部可视化完成")
